In [ ]:
"""
Main workflow:
1. Set project structure.
2. Merge the cleaned modelling CSV with the filtered neighborhood shapefile.
3. Report possible shapefile field-name truncation caused by the 10-character limit.
4. Calculate neighborhood-level median winter LST from the full LST raster.
5. Calculate Spearman correlation between REPC and neighborhood-level LST.
6. Calculate Spearman correlations between selected landscape metrics and neighborhood-level LST.

Required input files:
- model/form_energy_final_cleaned.csv
- shp/10580_filtered_neighborhood_boundariesx.shp
- data/NL_LST_2022_winter_m_3.tif

Main outputs:
- shp/merged_cleaned_shapefile.shp
- model/Spearman_REPC_LST_by_ste_mvs_and_all.csv
- model/Median_REPC_LST_by_ste_mvs.csv
- model/Spearman_neighborhood_LST_landscape_metrics.csv
"""

import os
import warnings
from typing import Dict, List, Tuple

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterstats import zonal_stats

warnings.filterwarnings("ignore", category=FutureWarning)


# ============================================================
# 0. PROJECT STRUCTURE
# ============================================================

BASE_DIR = os.getcwd()

DATA_DIR = os.path.join(BASE_DIR, "data")
OUT_DIR = os.path.join(BASE_DIR, "output")
WEIGHT_DIR = os.path.join(BASE_DIR, "weights")
SRC_DIR = os.path.join(BASE_DIR, "src")
SHP_DIR = os.path.join(BASE_DIR, "shp")
MODEL_DIR = os.path.join(BASE_DIR, "model")
FIGURE_DIR = os.path.join(BASE_DIR, "figures")

for directory in [DATA_DIR, OUT_DIR, WEIGHT_DIR, SRC_DIR, SHP_DIR, MODEL_DIR, FIGURE_DIR]:
    os.makedirs(directory, exist_ok=True)

print("PROJECT STRUCTURE READY")
print("BASE_DIR:", BASE_DIR)


# ============================================================
# 1. GLOBAL SETTINGS
# ============================================================

CLEANED_CSV = os.path.join(MODEL_DIR, "form_energy_final_cleaned.csv")
FILTERED_SHP = os.path.join(SHP_DIR, "10580_filtered_neighborhood_boundariesx.shp")
MERGED_SHP = os.path.join(SHP_DIR, "merged_cleaned_shapefile.shp")

COMM_ID_COL = "buurtcode"
STE_COL = "ste_mvs"

REPC_CANDIDATES = ["ec_inten_percapita", "ec_inten_p"]

FULL_LST_TIF = os.path.join(DATA_DIR, "NL_LST_2022_winter_m_3.tif")
FULL_LST_FIELD = "LST_median_2022"


# ============================================================
# Significant landscape metrics from SDM
# ============================================================

SIGNIFICANT_LM_CANDIDATES = {
    "AI_water": ["AI_water"],
    "IJI_water": ["IJI_water"],
    "AI_built": ["AI_built"],
    "IJI_built": ["IJI_built"],
    "PLAND_Dense_Short": ["PLAND_Dense_Short", "PLAND_Dens"],
    "PLAND_Dense_Tall": ["PLAND_Dense_Tall", "PLAND_De_1"],
    "AI_Dense_Tall": ["AI_Dense_Tall", "AI_Dense_T"],
}


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def require_file(path: str, label: str) -> None:
    """Raise an error if a required file does not exist."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required {label} not found: {path}")


def find_existing_column(df: pd.DataFrame, candidates: List[str], label: str) -> str:
    """Return the first existing column from a list of candidate names."""
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"Could not find {label}. Tried: {candidates}")


def truncate_shapefile_field(name: str) -> str:
    """Return the 10-character shapefile version of a field name."""
    return str(name)[:10]


def print_shapefile_truncation_report(csv_cols: List[str], shp_cols: List[str]) -> None:
    """Print possible mapping between original CSV names and truncated shapefile names."""
    mapping: Dict[str, List[str]] = {}
    for col in csv_cols:
        truncated = truncate_shapefile_field(col)
        mapping.setdefault(truncated, []).append(col)

    print("\n=== Possible Truncation Mapping (shapefile field → CSV field) ===")
    for shp_col in shp_cols:
        if shp_col in mapping:
            print(f"{shp_col}  ←  {mapping[shp_col]}")


def merge_csv_with_shapefile(
    csv_path: str = CLEANED_CSV,
    shp_path: str = FILTERED_SHP,
    output_path: str = MERGED_SHP
) -> gpd.GeoDataFrame:
    """Merge cleaned modelling CSV into the filtered neighborhood shapefile."""
    require_file(csv_path, "cleaned model CSV")
    require_file(shp_path, "filtered shapefile")

    df = pd.read_csv(csv_path, dtype={COMM_ID_COL: str})
    gdf = gpd.read_file(shp_path)
    gdf[COMM_ID_COL] = gdf[COMM_ID_COL].astype(str)

    merged = gdf.merge(df, on=COMM_ID_COL, how="inner")

    print_shapefile_truncation_report(df.columns.tolist(), merged.columns.tolist())

    merged.to_file(output_path, encoding="utf-8")
    print(f"\nSaved merged shapefile to:\n{output_path}")

    return merged


def calculate_median_lst(
    gdf: gpd.GeoDataFrame,
    raster_path: str,
    output_field: str,
    count_field: str = "n_pix_LST"
) -> gpd.GeoDataFrame:
    """Calculate median LST for each polygon using zonal statistics."""
    require_file(raster_path, "LST raster")

    gdf_out = gdf.copy()

    with rasterio.open(raster_path) as src:
        arr = src.read(1)
        transform = src.transform
        nodata = src.nodata

    if nodata is not None:
        arr = np.where(arr == nodata, np.nan, arr)

    arr = np.where(arr == 0, np.nan, arr)

    zs = zonal_stats(
        gdf_out,
        arr,
        affine=transform,
        stats=["median", "count"],
        nodata=np.nan
    )

    gdf_out[output_field] = [z["median"] for z in zs]
    gdf_out[count_field] = [z["count"] for z in zs]

    print(f"Computed {output_field}")
    return gdf_out


def spearman_rho(df: pd.DataFrame, x_col: str, y_col: str, min_n: int = 5) -> Tuple[float, int]:
    """Calculate Spearman rho between two columns."""
    sub = df[[x_col, y_col]].copy()
    sub[x_col] = pd.to_numeric(sub[x_col], errors="coerce")
    sub[y_col] = pd.to_numeric(sub[y_col], errors="coerce")
    sub = sub.dropna()

    n = len(sub)
    if n < min_n:
        return np.nan, n

    rho = sub[[x_col, y_col]].corr(method="spearman").iloc[0, 1]
    return float(rho), n


def spearman_all_and_by_urbanity(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    group_col: str = STE_COL,
    min_n: int = 5
) -> pd.DataFrame:
    """Calculate Spearman rho for all neighborhoods and by urbanity category."""
    work = df.copy()
    work[group_col] = pd.to_numeric(work[group_col], errors="coerce")
    work[x_col] = pd.to_numeric(work[x_col], errors="coerce")
    work[y_col] = pd.to_numeric(work[y_col], errors="coerce")

    work = work.dropna(subset=[group_col, x_col, y_col])

    rows = []

    rho_all, n_all = spearman_rho(work, x_col, y_col, min_n=min_n)
    rows.append({
        "group": "ALL",
        "n": n_all,
        "x_variable": x_col,
        "y_variable": y_col,
        "spearman_rho": rho_all
    })

    print(f"\nSpearman rho: {x_col} vs {y_col}")
    print(f"ALL: rho = {rho_all:.3f} (n={n_all})")

    for urbanity in sorted(work[group_col].dropna().unique()):
        group_df = work[work[group_col] == urbanity]
        rho_group, n_group = spearman_rho(group_df, x_col, y_col, min_n=min_n)

        rows.append({
            "group": f"ste_mvs={int(urbanity)}",
            "n": n_group,
            "x_variable": x_col,
            "y_variable": y_col,
            "spearman_rho": rho_group
        })

    return pd.DataFrame(rows)


# ============================================================
# 3. FULL LST VS REPC ANALYSIS
# ============================================================

def run_repc_lst_analysis(gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """Analyze Spearman correlation between REPC and neighborhood-level median LST."""
    print("\n" + "=" * 80)
    print("FULL LST vs REPC ANALYSIS")
    print("=" * 80)

    repc_col = find_existing_column(gdf, REPC_CANDIDATES, "REPC column")

    gdf_lst = calculate_median_lst(
        gdf=gdf,
        raster_path=FULL_LST_TIF,
        output_field=FULL_LST_FIELD,
        count_field="n_pix_LST_full"
    )

    df_corr = gdf_lst.dropna(subset=[FULL_LST_FIELD, repc_col, STE_COL]).copy()

    results = spearman_all_and_by_urbanity(
        df=df_corr,
        x_col=repc_col,
        y_col=FULL_LST_FIELD
    )

    output_path = os.path.join(MODEL_DIR, "Spearman_REPC_LST_by_ste_mvs_and_all.csv")
    results.to_csv(output_path, index=False, encoding="utf-8-sig")

    group_median = (
        df_corr
        .groupby(STE_COL)[[repc_col, FULL_LST_FIELD]]
        .median()
        .reset_index()
    )

    group_median_output = os.path.join(MODEL_DIR, "Median_REPC_LST_by_ste_mvs.csv")
    group_median.to_csv(group_median_output, index=False, encoding="utf-8-sig")

    return results


# ============================================================
# 4. NEIGHBORHOOD-LEVEL LST VS LANDSCAPE METRICS
# ============================================================

def run_neighborhood_lst_metric_analysis(gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    """
    Calculate Spearman correlations between neighborhood-level median LST
    and landscape metrics with significant direct impacts on REPC.
    """
    print("\n" + "=" * 80)
    print("NEIGHBORHOOD-LEVEL LST vs LANDSCAPE METRICS")
    print("=" * 80)

    gdf_lst = calculate_median_lst(
        gdf=gdf,
        raster_path=FULL_LST_TIF,
        output_field="LST_neighborhood_median",
        count_field="n_pix_LST_full"
    )

    rows = []

    for metric_label, candidates in SIGNIFICANT_LM_CANDIDATES.items():
        try:
            metric_col = find_existing_column(gdf_lst, candidates, metric_label)
        except ValueError:
            print(f"Skipping {metric_label}: column not found.")
            continue

        grouped_result = spearman_all_and_by_urbanity(
            df=gdf_lst,
            x_col=metric_col,
            y_col="LST_neighborhood_median"
        )

        for _, r in grouped_result.iterrows():
            rows.append({
                "metric_label": metric_label,
                "metric_column": metric_col,
                "lst_variable": "LST_neighborhood_median",
                "group": r["group"],
                "n": r["n"],
                "spearman_rho": r["spearman_rho"]
            })

    result_df = pd.DataFrame(rows)

    output_path = os.path.join(
        MODEL_DIR,
        "Spearman_neighborhood_LST_landscape_metrics.csv"
    )
    result_df.to_csv(output_path, index=False, encoding="utf-8-sig")

    print("\nSaved neighborhood LST-landscape metric Spearman results:")
    print(output_path)

    print("\nALL-neighborhood results only:")
    print(result_df[result_df["group"] == "ALL"])

    return result_df


# ============================================================
# 5. RUN ALL ANALYSES
# ============================================================

if __name__ == "__main__":
    merged_gdf = merge_csv_with_shapefile(
        csv_path=CLEANED_CSV,
        shp_path=FILTERED_SHP,
        output_path=MERGED_SHP
    )

    # Reload saved shapefile (preserves truncated field names)
    merged_gdf = gpd.read_file(MERGED_SHP)

    run_repc_lst_analysis(merged_gdf)
    run_neighborhood_lst_metric_analysis(merged_gdf)

    print("\nALL NEIGHBORHOOD-LEVEL LST ANALYSIS STEPS COMPLETED SUCCESSFULLY!")